# NIFTY Gap Strategy — v10 (Regime Gates + Optional Wider SL/TP)

Three improvements over v7/v8:

| Problem | Fix |
|---|---|
| Gap-reversal signal applied in momentum regimes | Hard gate: skip when `\|gap_normalized\| > 0.8` |
| Trading when India VIX is expensive / high-vol | Hard gate: skip when `VIX_INDIA_pct > 65th percentile` |
| No persistence in regime detection | HMM (2-state) on NIFTY returns: skip when `P(high-vol) > 0.60` |

**Phase 1 (this notebook):** Apply gates to existing `sim_cache.csv` — no re-simulation needed.

**Phase 2 (resim_v10.ipynb):** Re-simulate with SL=10%/TP=60% → new cache → re-run analysis.

**Key diagnostic:** Step through quarterly base rates before and after gates to confirm the gates eliminate bad-quarter trades without hurting good-quarter trades.

In [18]:
import warnings
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from datetime import date
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from hmmlearn.hmm import GaussianHMM

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Fixed trade parameters ────────────────────────────────────────────────────
LOT_SIZE         = 75
BASE_LOTS        = 5
MAX_LOTS         = 25
DTE0_MAX_LOTS    = 10
STARTING_CAPITAL = 200_000.0

# ── SL/TP/BREAKEVEN/KELLY_ODDS — auto-detected in cell-2 from SIM_CACHE_PATH ─
# Phase 1 (v6 cache): SL=15%, TP=40%, BE=27.3%, Kelly odds=2.667
# Phase 2 (v10 cache): SL=10%, TP=60%, BE=14.3%, Kelly odds=6.0

# ── Walk-forward split (same as v7/v8) ────────────────────────────────────────
TRAIN_END = date(2025, 6, 30)
OOS_START = date(2025, 7, 1)

# ── Regime gate thresholds ────────────────────────────────────────────────────
GAP_NORM_MAX    = 0.80
GAP_NORM_MIN    = 0.10
VIX_PCT_MAX     = 0.65
HMM_STATES      = 2
HMM_HIGHVOL_THR = 0.60

print('Config loaded.')

Config loaded.


In [19]:
# ── Paths ─────────────────────────────────────────────────────────────────────
GAP_TRADING    = Path.cwd().parent
SIM_CACHE_PATH = GAP_TRADING / 'v6' / 'sim_cache.csv'
ALIGNED_CSV    = GAP_TRADING / 'v2' / 'v2_aligned_dataset.csv'
V7_MODEL_PATH  = GAP_TRADING / 'v7' / 'v7_model.pkl'

for lbl, p in [('sim_cache', SIM_CACHE_PATH), ('aligned CSV', ALIGNED_CSV), ('v7 model', V7_MODEL_PATH)]:
    print(f'{lbl:<12}: {"OK" if p.exists() else "MISSING"}  ({p})')

# ── Auto-detect SL/TP from cache name ────────────────────────────────────────
if 'v10' in str(SIM_CACHE_PATH):
    SL_PCT, TP_PCT = 0.10, 0.60
else:
    SL_PCT, TP_PCT = 0.15, 0.40
BREAKEVEN  = SL_PCT / (SL_PCT + TP_PCT)
KELLY_ODDS = TP_PCT / SL_PCT
print(f'\nCache phase  : {"v10 (SL=10%/TP=60%)" if SL_PCT == 0.10 else "v6  (SL=15%/TP=40%)"}')
print(f'Breakeven    : {BREAKEVEN:.1%}')
print(f'Kelly odds   : {KELLY_ODDS:.1f}x')

# ── Load sim_cache ────────────────────────────────────────────────────────────
sim_df = pd.read_csv(SIM_CACHE_PATH, parse_dates=['date'])
sim_df['date'] = sim_df['date'].dt.date
sim_core = sim_df[['date', 'win', 'exit_reason', 'entry_prem', 'exit_prem', 'dte']].copy()
print(f'\nsim_cache : {len(sim_df)} rows  ({sim_df["date"].min()} → {sim_df["date"].max()})')

# ── Load aligned dataset ──────────────────────────────────────────────────────
aligned = pd.read_csv(ALIGNED_CSV, parse_dates=['india_date'])
aligned['india_date'] = aligned['india_date'].dt.date
aligned['VIX_INDIA_level'] = aligned['VIX_INDIA_level'].ffill().bfill()
aligned = aligned.sort_values('india_date').reset_index(drop=True)
print(f'aligned   : {len(aligned)} rows  ({aligned["india_date"].min()} → {aligned["india_date"].max()})')

# ── Rolling features ──────────────────────────────────────────────────────────
aligned['nifty_20d_realized_vol'] = aligned['prev_india_ret'].rolling(20).std()
aligned['nifty_20d_ret'] = (
    np.exp(np.log1p(aligned['prev_india_ret']).rolling(20).sum()) - 1
)
aligned['VIX_INDIA_pct'] = (
    aligned['VIX_INDIA_level']
    .rolling(252, min_periods=60)
    .rank(pct=True)
)

ALIGNED_COLS = [
    'india_date', 'gap_pct', 'prev_india_ret',
    'SP500_ret', 'NASDAQ_ret', 'DOW_ret',
    'DAX_ret', 'FTSE_ret',
    'NIKKEI_ret', 'HANGSENG_ret', 'SGX_ret',
    'VIX_US_ret', 'VIX_US_level', 'VIX_INDIA_level',
    'nifty_20d_ret', 'nifty_20d_realized_vol', 'VIX_INDIA_pct',
]

merged = (sim_core
          .merge(aligned[ALIGNED_COLS], left_on='date', right_on='india_date', how='inner')
          .drop(columns=['india_date']))

merged['us_ret']         = merged[['SP500_ret', 'NASDAQ_ret', 'DOW_ret']].mean(axis=1)
merged['europe_ret']     = merged[['DAX_ret', 'FTSE_ret']].mean(axis=1)
merged['asia_ret']       = merged[['NIKKEI_ret', 'HANGSENG_ret', 'SGX_ret']].mean(axis=1)
merged['log_entry_prem'] = np.log(merged['entry_prem'].clip(lower=0.1))
merged['gap_normalized'] = merged['gap_pct'] / merged['nifty_20d_realized_vol'].replace(0, np.nan)
merged['month_key']      = merged['date'].apply(lambda d: (d.year, d.month))
merged['quarter']        = merged['date'].apply(lambda d: f'{d.year}Q{(d.month-1)//3+1}')

gate_cols = ['gap_normalized', 'VIX_INDIA_pct', 'nifty_20d_realized_vol']
before = len(merged)
merged = merged.dropna(subset=gate_cols).reset_index(drop=True)
merged = merged.sort_values('date').reset_index(drop=True)

print(f'merged    : {len(merged)} rows  (dropped {before-len(merged)} NaN rows from rolling warmup)')

print(f'\nQuarterly base win rates  (breakeven = {BREAKEVEN:.1%}):')
q_tbl = merged.groupby('quarter').agg(N=('win','count'), WR=('win','mean'))
q_tbl['WR_str']   = q_tbl['WR'].map('{:.1%}'.format)
q_tbl['Below_BE'] = q_tbl['WR'].map(lambda v: '← BELOW BE' if v < BREAKEVEN else '')
print(q_tbl[['N','WR_str','Below_BE']].to_string())

sim_cache   : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v6\sim_cache.csv)
aligned CSV : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v2\v2_aligned_dataset.csv)
v7 model    : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v7\v7_model.pkl)

Cache phase  : v6  (SL=15%/TP=40%)
Breakeven    : 27.3%
Kelly odds   : 2.7x

sim_cache : 402 rows  (2024-01-02 → 2026-03-24)
aligned   : 740 rows  (2023-03-31 → 2026-04-02)
merged    : 402 rows  (dropped 0 NaN rows from rolling warmup)

Quarterly base win rates  (breakeven = 27.3%):
          N WR_str    Below_BE
quarter                       
2024Q1   47  29.8%            
2024Q2   44  27.3%            
2024Q3   46  17.4%  ← BELOW BE
2024Q4   42  26.2%  ← BELOW BE
2025Q1   48  27.1%  ← BELOW BE
2025Q2   42  14.3%  ← BELOW BE
2025Q3   49  22.4%  ← BELOW BE
2025Q4   43  27.9%            
2026Q1   41  31.7%    

In [20]:
# ── HMM regime detection ──────────────────────────────────────────────────────
hmm_input = aligned['prev_india_ret'].dropna()
hmm_dates = hmm_input.index.map(lambda i: aligned.loc[i, 'india_date'])
X_hmm = hmm_input.values.reshape(-1, 1)

hmm = GaussianHMM(n_components=HMM_STATES, covariance_type='full',
                  n_iter=300, random_state=42)
hmm.fit(X_hmm)

state_vars     = [float(hmm.covars_[s][0, 0]) for s in range(HMM_STATES)]
HIGH_VOL_STATE = int(np.argmax(state_vars))
state_means    = [float(hmm.means_[s][0]) for s in range(HMM_STATES)]

print(f'HMM fitted on {len(X_hmm)} daily NIFTY returns')
print(f'  State variances : {[round(v, 6) for v in state_vars]}')
print(f'  State means     : {[round(m, 5) for m in state_means]}')
print(f'  High-vol state  : {HIGH_VOL_STATE}  (larger variance)')
print(f'  Transition matrix:\n{hmm.transmat_.round(3)}')

_, posteriors = hmm.score_samples(X_hmm)
hmm_df = pd.DataFrame({
    'date'          : hmm_dates.values,   # matches merged['date']
    'hmm_state'     : hmm.predict(X_hmm),
    'hmm_p_highvol' : posteriors[:, HIGH_VOL_STATE],
})

merged = merged.merge(hmm_df, on='date', how='left')
merged['hmm_p_highvol'] = merged['hmm_p_highvol'].fillna(0.5)

print(f'\nHMM state distribution in merged data ({len(merged)} rows):')
vc = merged['hmm_state'].value_counts()
for s, cnt in vc.items():
    label = 'HIGH-VOL' if s == HIGH_VOL_STATE else 'low-vol '
    print(f'  State {s} ({label}): {cnt} days  ({cnt/len(merged):.1%})')

print(f'\nP(high-vol) by quarter:')
print(merged.groupby('quarter')['hmm_p_highvol'].agg(['mean','max']).round(3).to_string())

HMM fitted on 740 daily NIFTY returns
  State variances : [5.9e-05, 0.000772]
  State means     : [0.00047, -0.00152]
  High-vol state  : 1  (larger variance)
  Transition matrix:
[[0.993 0.007]
 [0.119 0.881]]

HMM state distribution in merged data (402 rows):
  State 0 (low-vol ): 383 days  (95.3%)
  State 1 (HIGH-VOL): 19 days  (4.7%)

P(high-vol) by quarter:
          mean    max
quarter              
2024Q1   0.003  0.022
2024Q2   0.079  1.000
2024Q3   0.008  0.216
2024Q4   0.004  0.030
2025Q1   0.002  0.011
2025Q2   0.150  0.989
2025Q3   0.001  0.001
2025Q4   0.001  0.001
2026Q1   0.213  1.000


In [21]:
# ── Gate definitions ──────────────────────────────────────────────────────────
def gate_gap_norm(df):
    """Skip momentum days: gap too large relative to vol, or too small to matter."""
    return (df['gap_normalized'].abs() <= GAP_NORM_MAX) & \
           (df['gap_normalized'].abs() >= GAP_NORM_MIN)

def gate_vix_pct(df):
    """Skip high-vol regime: India VIX above 65th rolling percentile."""
    return df['VIX_INDIA_pct'] <= VIX_PCT_MAX

def gate_hmm(df):
    """Skip sustained high-vol state: HMM P(high-vol) > threshold."""
    return df['hmm_p_highvol'] <= HMM_HIGHVOL_THR

def apply_gates(df, use_gap=True, use_vix=True, use_hmm=True):
    mask = pd.Series(True, index=df.index)
    if use_gap: mask &= gate_gap_norm(df)
    if use_vix: mask &= gate_vix_pct(df)
    if use_hmm: mask &= gate_hmm(df)
    return mask

# ── DIAGNOSTIC 1: Quarterly breakdown — core analysis ────────────────────────
print('=' * 75)
print('  QUARTERLY GATE ANALYSIS — does filtering improve bad quarters?')
print('=' * 75)
print(f'\n{"Quarter":<9} {"N_all":>5} {"WR_all":>7} {"N_gap":>6} {"WR_gap":>7} '
      f'{"N_vix":>6} {"WR_vix":>7} {"N_all3":>6} {"WR_all3":>7}')
print('-' * 75)

for q in sorted(merged['quarter'].unique()):
    qdf = merged[merged['quarter'] == q].copy()
    n_all = len(qdf)
    wr_all = qdf['win'].mean()

    g_gap  = gate_gap_norm(qdf); n_gap  = g_gap.sum();  wr_gap  = qdf[g_gap]['win'].mean()  if g_gap.sum()  > 0 else float('nan')
    g_vix  = gate_vix_pct(qdf); n_vix  = g_vix.sum();  wr_vix  = qdf[g_vix]['win'].mean()  if g_vix.sum()  > 0 else float('nan')
    g_all3 = apply_gates(qdf);  n_all3 = g_all3.sum(); wr_all3 = qdf[g_all3]['win'].mean() if g_all3.sum() > 0 else float('nan')

    flag = ' ← BAD' if wr_all < BREAKEVEN else ''
    print(f'{q:<9} {n_all:>5} {wr_all:>7.1%} {n_gap:>6} {wr_gap:>7.1%} '
          f'{n_vix:>6} {wr_vix:>7.1%} {n_all3:>6} {wr_all3:>7.1%}{flag}')

print(f'\nBreakeven = {BREAKEVEN:.1%}')

# ── DIAGNOSTIC 2: Per-gate OOS contribution ───────────────────────────────────
oos_all = merged[merged['date'] >= OOS_START].copy()
train_all = merged[merged['date'] <= TRAIN_END].copy()

print(f'\n{"─"*60}')
print(f'OOS ({OOS_START}+): {len(oos_all)} total days  |  base WR = {oos_all["win"].mean():.1%}')
print(f'{"─"*60}')
print(f'{"Gate combination":<30} {"N":>4} {"WR":>7} {"% filtered":>11}')
print(f'{"─"*60}')

for name, kwargs in [
    ('No gates (baseline)',          dict(use_gap=False, use_vix=False, use_hmm=False)),
    ('gap_norm only',                dict(use_gap=True,  use_vix=False, use_hmm=False)),
    ('VIX pct only',                 dict(use_gap=False, use_vix=True,  use_hmm=False)),
    ('HMM only',                     dict(use_gap=False, use_vix=False, use_hmm=True)),
    ('gap_norm + VIX pct',           dict(use_gap=True,  use_vix=True,  use_hmm=False)),
    ('gap_norm + HMM',               dict(use_gap=True,  use_vix=False, use_hmm=True)),
    ('VIX pct + HMM',                dict(use_gap=False, use_vix=True,  use_hmm=True)),
    ('ALL THREE GATES',              dict(use_gap=True,  use_vix=True,  use_hmm=True)),
]:
    m   = apply_gates(oos_all, **kwargs)
    sub = oos_all[m]
    wr  = sub['win'].mean() if len(sub) > 0 else float('nan')
    pct_kept    = len(sub) / len(oos_all) * 100
    pct_filtered = 100 - pct_kept
    print(f'{name:<30} {len(sub):>4} {wr:>7.1%} {pct_filtered:>10.1f}%')

# ── DIAGNOSTIC 3: What are gate feature values in bad vs good quarters? ───────
print(f'\n{"─"*60}')
print('Gate feature means by quarter (helps verify mechanism):')
diag_cols = ['gap_normalized', 'VIX_INDIA_pct', 'hmm_p_highvol', 'VIX_INDIA_level']
print(merged.groupby('quarter')[diag_cols].mean().round(3).to_string())

  QUARTERLY GATE ANALYSIS — does filtering improve bad quarters?

Quarter   N_all  WR_all  N_gap  WR_gap  N_vix  WR_vix N_all3 WR_all3
---------------------------------------------------------------------------
2024Q1       47   29.8%     34   29.4%      2    0.0%      1    0.0%
2024Q2       44   27.3%     30   26.7%     13   30.8%     10   30.0%
2024Q3       46   17.4%     31   12.9%     31    6.5%     20    0.0% ← BAD
2024Q4       42   26.2%     29   31.0%     24   33.3%     18   38.9% ← BAD
2025Q1       48   27.1%     34   35.3%     28   25.0%     18   33.3% ← BAD
2025Q2       42   14.3%     25   12.0%     15   13.3%      9   11.1% ← BAD
2025Q3       49   22.4%     34   23.5%     49   22.4%     34   23.5% ← BAD
2025Q4       43   27.9%     32   34.4%     43   27.9%     32   34.4%
2026Q1       41   31.7%     27   22.2%     24   33.3%     17   23.5%

Breakeven = 27.3%

────────────────────────────────────────────────────────────
OOS (2025-07-01+): 133 total days  |  base WR = 27.1%
───

In [22]:
# ── Load v7 model + score all rows ───────────────────────────────────────────
bundle   = joblib.load(V7_MODEL_PATH)
model    = bundle['model']
scaler   = bundle['scaler']
thr      = bundle['threshold']
FEATURES = bundle['features']

print(f'v7 model loaded:')
print(f'  Features   : {FEATURES}')
print(f'  Threshold  : {thr}')
print(f'  Train AUC  : {bundle["train_auc"]}')
print(f'  Best C     : {bundle["best_C"]}')

# Impute any NaNs in FEATURES with column median (handles nifty_20d_ret warmup rows)
X_all = merged[FEATURES].copy()
nan_counts = X_all.isna().sum()
if nan_counts.any():
    print(f'\nNaN counts in FEATURES before imputation:')
    print(nan_counts[nan_counts > 0].to_string())
    X_all = X_all.fillna(X_all.median())

merged['prob_win'] = model.predict_proba(scaler.transform(X_all.values))[:, 1]
print(f'\nScored {len(merged)} rows. prob_win range: [{merged["prob_win"].min():.3f}, {merged["prob_win"].max():.3f}]')

# ── Backtest engine ───────────────────────────────────────────────────────────
def round_trip_charges(ep, xp, lots):
    bv = ep * lots * LOT_SIZE; sv = xp * lots * LOT_SIZE
    return round(20*2 + 0.00003*bv + 0.000625*sv + 0.00053*(bv+sv) + 0.000001*(bv+sv) + 0.18*(40+0.00053*(bv+sv)+0.000001*(bv+sv)), 2)

def run_backtest(label, subset, gate_mask=None, use_kelly=False):
    if gate_mask is None:
        gate_mask = pd.Series(True, index=subset.index)
    tradeable = subset[gate_mask & (subset['prob_win'] >= thr)].copy()
    skipped_gate = (~gate_mask).sum()
    skipped_prob = (gate_mask & (subset['prob_win'] < thr)).sum()

    if len(tradeable) == 0:
        print(f'\n{label}: 0 trades (gate filtered {skipped_gate}, prob filtered {skipped_prob}).')
        return pd.DataFrame()

    capital, peak, rows = STARTING_CAPITAL, STARTING_CAPITAL, []
    for _, row in tradeable.iterrows():
        ep, xp, dte = float(row['entry_prem']), float(row['exit_prem']), int(row['dte'])
        prob = float(row['prob_win'])
        cost_lot = ep * LOT_SIZE
        if cost_lot <= 0: continue
        if use_kelly:
            f    = max(0.0, prob - (1-prob)/KELLY_ODDS)
            lots = max(1, int(f * capital / cost_lot))
        else:
            lots = max(BASE_LOTS, int(capital // cost_lot))
        lots = min(lots, MAX_LOTS)
        if dte == 0: lots = min(lots, DTE0_MAX_LOTS)
        chg = round_trip_charges(ep, xp, lots)
        pnl = (xp - ep) * LOT_SIZE * lots - chg
        capital += pnl; peak = max(peak, capital)
        rows.append({'Date': row['date'], 'Quarter': row['quarter'],
                     'P(win)': round(prob,3), 'DTE': dte, 'Lots': lots,
                     'Entry': ep, 'Exit': xp, 'PnL(pts)': round(xp-ep,2),
                     'Trade PnL': round(pnl,2), 'Capital': round(capital,2),
                     'DD%': round((peak-capital)/peak*100,2),
                     'Reason': row['exit_reason']})

    ledger = pd.DataFrame(rows)
    wins   = (ledger['Trade PnL'] > 0).sum()
    total  = len(ledger)
    roi    = (capital - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    maxdd  = ledger['DD%'].max()
    aw     = ledger.loc[ledger['Trade PnL']>0,'Trade PnL'].mean() if wins>0 else 0
    al     = ledger.loc[ledger['Trade PnL']<=0,'Trade PnL'].mean() if wins<total else 0

    print(f'\n{"="*62}')
    print(f'  {label}')
    print(f'{"="*62}')
    print(f'  Gate filtered (not assessed)    : {skipped_gate}')
    print(f'  Prob filtered (gate pass, low p): {skipped_prob}')
    print(f'  Traded                          : {total}')
    print(f'  Win rate : {wins/total*100:.1f}%  (breakeven: {BREAKEVEN:.1%})')
    print(f'  ROI      : {roi:+.1f}%')
    print(f'  Max DD   : {maxdd:.1f}%')
    print(f'  Avg win  : Rs {aw:,.0f}  |  Avg loss: Rs {al:,.0f}')
    print(f'{"="*62}')
    print(ledger['Reason'].value_counts().to_string())
    print()
    print(ledger[['Date','Quarter','P(win)','DTE','Lots','Entry','Exit',
                  'PnL(pts)','Trade PnL','Capital','DD%','Reason']].to_string(index=False))
    return ledger

# ── Run all gate combinations on OOS ─────────────────────────────────────────
oos = merged[merged['date'] >= OOS_START].copy()

ledger_no_gates   = run_backtest('v10 OOS — No gates  (v7 model only)', oos,
                                 gate_mask=pd.Series(True, index=oos.index))
ledger_gap_only   = run_backtest('v10 OOS — gap_norm gate only', oos,
                                 gate_mask=gate_gap_norm(oos))
ledger_vix_only   = run_backtest('v10 OOS — VIX pct gate only', oos,
                                 gate_mask=gate_vix_pct(oos))
ledger_all3       = run_backtest('v10 OOS — ALL THREE GATES', oos,
                                 gate_mask=apply_gates(oos))
ledger_all3_kelly = run_backtest('v10 OOS — ALL THREE GATES + Kelly sizing', oos,
                                 gate_mask=apply_gates(oos), use_kelly=True)

v7 model loaded:
  Features   : ['gap_pct', 'prev_india_ret', 'us_ret', 'europe_ret', 'asia_ret', 'VIX_US_ret', 'VIX_US_level', 'VIX_INDIA_level', 'log_entry_prem', 'dte', 'nifty_20d_ret', 'nifty_20d_realized_vol', 'gap_normalized']
  Threshold  : 0.58
  Train AUC  : 0.667
  Best C     : 0.5

NaN counts in FEATURES before imputation:
us_ret          17
europe_ret       4
VIX_US_ret      17
VIX_US_level    17

Scored 402 rows. prob_win range: [0.028, 0.987]

  v10 OOS — No gates  (v7 model only)
  Gate filtered (not assessed)    : 0
  Prob filtered (gate pass, low p): 122
  Traded                          : 11
  Win rate : 45.5%  (breakeven: 27.3%)
  ROI      : +11.3%
  Max DD   : 21.0%
  Avg win  : Rs 32,527  |  Avg loss: Rs -23,345
Reason
Target Hit    5
Stop Loss     4
11:15 exit    2

      Date Quarter  P(win)  DTE  Lots  Entry   Exit  PnL(pts)  Trade PnL   Capital   DD%     Reason
2025-07-02  2025Q3   0.611    1    25  56.40  78.96     22.56   41998.07 241998.07  0.00 Target Hit
2

In [23]:
# ── Summary comparison table ──────────────────────────────────────────────────
def _row(ledger, label):
    if ledger.empty:
        return {'Strategy': label, 'Trades': 0, 'Win%': 'N/A', 'ROI': 'N/A', 'MaxDD': 'N/A'}
    wins  = (ledger['Trade PnL'] > 0).sum()
    total = len(ledger)
    roi   = (ledger['Capital'].iloc[-1] - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    maxdd = ledger['DD%'].max()
    return {'Strategy': label, 'Trades': total,
            'Win%': f'{wins/total*100:.1f}%', 'ROI': f'{roi:+.1f}%', 'MaxDD': f'{maxdd:.1f}%'}

rows = [
    {'Strategy': 'v7  (no gates, threshold=0.58)',    'Trades': 10, 'Win%': '40.0%', 'ROI': '+8.4%',  'MaxDD': '21.0%'},
    {'Strategy': 'v8  L1 walk-forward (no gates)',    'Trades':  7, 'Win%': '28.6%', 'ROI': '-19.5%', 'MaxDD': '24.8%'},
    _row(ledger_no_gates,    'v10 no gates   (v7 model, reloaded)'),
    _row(ledger_gap_only,    'v10 gap_norm gate only'),
    _row(ledger_vix_only,    'v10 VIX pct gate only'),
    _row(ledger_all3,        'v10 ALL THREE gates (fixed sizing)'),
    _row(ledger_all3_kelly,  'v10 ALL THREE gates + Kelly sizing'),
]

print()
print('=' * 70)
print('  v10 REGIME-GATED BACKTEST — SUMMARY')
print('=' * 70)
print(pd.DataFrame(rows).to_string(index=False))
print('=' * 70)
print(f'\nGate thresholds used:')
print(f'  gap_norm: |gap_normalized| ∈ [{GAP_NORM_MIN}, {GAP_NORM_MAX}]')
print(f'  VIX pct : VIX_INDIA_pct ≤ {VIX_PCT_MAX}  (rolling 252-day window)')
print(f'  HMM     : P(high-vol state) ≤ {HMM_HIGHVOL_THR}  (2-state GaussianHMM)')
print(f'\nv7 model threshold : {thr}  |  Breakeven : {BREAKEVEN:.1%}  |  Kelly odds : {KELLY_ODDS:.1f}x')
cache_phase = 'v10 (SL=10%/TP=60%)' if SL_PCT == 0.10 else 'v6 (SL=15%/TP=40%)'
print(f'Cache used         : {SIM_CACHE_PATH.name}  [{cache_phase}]')
print(f'NOTE: v7 model was trained on v6 outcomes (SL=15%/TP=40%).')
print(f'      Scores predict P(TP hit under old rules) — misaligned with {cache_phase} cache.')
print(f'      For a properly aligned model, retrain on {SIM_CACHE_PATH.name} outcomes (v11).')

# ── Export results ────────────────────────────────────────────────────────────
OUT = Path.cwd()
cache_tag = 'v10' if 'v10' in str(SIM_CACHE_PATH) else 'v6'

summary_df = pd.DataFrame(rows)
summary_df.to_csv(OUT / f'results_{cache_tag}_summary.csv', index=False)
print(f'\nExported: results_{cache_tag}_summary.csv')

ledger_map = {
    f'results_{cache_tag}_oos_no_gates.csv'   : ledger_no_gates,
    f'results_{cache_tag}_oos_gap_only.csv'   : ledger_gap_only,
    f'results_{cache_tag}_oos_vix_only.csv'   : ledger_vix_only,
    f'results_{cache_tag}_oos_all3_gates.csv' : ledger_all3,
    f'results_{cache_tag}_oos_all3_kelly.csv' : ledger_all3_kelly,
}
for fname, ledger in ledger_map.items():
    if not ledger.empty:
        ledger.to_csv(OUT / fname, index=False)
        print(f'Exported: {fname}  ({len(ledger)} trades)')


  v10 REGIME-GATED BACKTEST — SUMMARY
                           Strategy  Trades   Win%    ROI MaxDD
     v7  (no gates, threshold=0.58)      10  40.0%  +8.4% 21.0%
     v8  L1 walk-forward (no gates)       7  28.6% -19.5% 24.8%
v10 no gates   (v7 model, reloaded)      11  45.5% +11.3% 21.0%
             v10 gap_norm gate only       4  50.0% +23.4% 17.9%
              v10 VIX pct gate only       5  60.0% +29.3% 21.0%
 v10 ALL THREE gates (fixed sizing)       2 100.0% +50.3%  0.0%
 v10 ALL THREE gates + Kelly sizing       2 100.0% +38.7%  0.0%

Gate thresholds used:
  gap_norm: |gap_normalized| ∈ [0.1, 0.8]
  VIX pct : VIX_INDIA_pct ≤ 0.65  (rolling 252-day window)
  HMM     : P(high-vol state) ≤ 0.6  (2-state GaussianHMM)

v7 model threshold : 0.58  |  Breakeven : 27.3%  |  Kelly odds : 2.7x
Cache used         : sim_cache.csv  [v6 (SL=15%/TP=40%)]
NOTE: v7 model was trained on v6 outcomes (SL=15%/TP=40%).
      Scores predict P(TP hit under old rules) — misaligned with v6 (SL=15%/TP=

In [24]:
# placeholder